# 2D ImageNet → 3D VGG16 weight transfer

Builds the 3D VGG16 backbone with ImageNet weights inflated to 3D via `classification-models-3D` (Solovyev et al., 2022), adds the custom dropout / batch-norm classification head, and runs a smoke test on random tensors to check shapes. This is the transfer-learning architecture shown on the poster.

> Requires approved ADNI data, which is not included in this repository. See the [project README](https://github.com/AlexKuaiR/AlzheimersFLAIR#readme).

In [ ]:
import tensorflow as tf
import keras
from keras.layers import Conv3D, MaxPooling3D, GlobalAveragePooling3D
from keras import Sequential
from keras.applications.vgg16 import VGG16
from keras import models
import numpy as np

In [ ]:
from classification_models_3D.tfkeras import Classifiers

VGG16, preprocess_input = Classifiers.get('vgg16')
conv_layers_pretrained = VGG16(input_shape=(32, 128, 128, 3), weights='imagenet')
# must have 32+ depth otherwise there will be negative depth

In [ ]:
conv_layers_pretrained.summary()

In [ ]:
global_average_layer = GlobalAveragePooling3D()

dense_layers = Sequential([
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1024, activation = 'relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(3, activation='softmax'),
])

model = Sequential([
    conv_layers_pretrained, global_average_layer, keras.layers.BatchNormalization(),
    dense_layers
])

In [ ]:
from keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate = 0.005, decay = 0.0012), 
              loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
model.summary()

In [ ]:
x_train = np.random.rand(300, 32, 128, 128, 3)
y_train = np.random.rand(300, 3)

In [ ]:

history=model.fit(x_train, y_train,
                        epochs = 100,
                        verbose = 1)
